# RepeatRadar Cohort Calculation Tutorial

## Welcome to Cohort Analysis with RepeatRadar!

This comprehensive tutorial demonstrates how to use the `repeatradar` package for cohort analysis. You'll learn to:

- 📊 Calculate user retention cohorts
- 💰 Analyze revenue patterns over time
- 📈 Compute retention rates and percentages
- 🎯 Filter data for segment-specific insights
- ⚡ Use flexible time periods and aggregation functions

Let's dive in!

In [1]:
# Install RepeatRadar if needed
# Uncomment the line below if you haven't installed the package yet
# !pip install repeatradar --upgrade

In [2]:
# Import the package and check version
import repeatradar

# Note: If you're working with the development version, version might not be available
try:
    print(f"RepeatRadar version: {repeatradar.__version__}")
except AttributeError:
    print("RepeatRadar is not installed or the version is not available. Install it using 'pip install repeatradar --upgrade'.")

RepeatRadar version: 0.6.0


In [3]:
from repeatradar import generate_cohort_data
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', None)

This notebook demonstrates the `generate_cohort_data` function usage:

## 🚀 Getting Started

### The Core Function: `generate_cohort_data()`

The `generate_cohort_data()` function is the heart of RepeatRadar. It transforms your transactional data into cohort analysis tables that reveal user behavior patterns over time.

**Key Features:**
- 🗓️ Flexible time periods (daily, weekly, monthly, quarterly, yearly)
- 📊 Multiple metrics (user counts, revenue, averages, unique values)
- 📈 Automatic retention rate calculations
- 🔧 Complete data with no missing periods
- 📋 Both pivot and long format outputs

Let's see it in action with real e-commerce data!

## 📊 Sample Dataset

We'll use a real e-commerce dataset containing customer transactions. This dataset includes:

- **CustomerID**: Unique customer identifier
- **InvoiceDateTime**: Transaction timestamp
- **TotalPrice**: Transaction value
- Additional transaction details for comprehensive analysis

In [7]:
# Load the sample e-commerce dataset
ecommerce_data = pd.read_pickle("https://github.com/krinya/repeatradar/raw/refs/heads/main/examples/data/ecommerce_data_2.pkl")
ecommerce_data.head()

,Order_Date,Time,Aging,Customer_Id,Gender,Device_Type,Customer_Login_type,Product_Category,Product,Sales,Quantity,Discount,Profit,Shipping_Cost,Order_Priority,Payment_method,OderedDateTime
0,2018-01-02,10:56:33,8.0,37077,Female,Web,Member,Auto & Accessories,Car Media Players,140.0,1.0,0.3,46.0,4.6,Medium,credit_card,2018-01-02 10:56:33
1,2018-07-24,20:41:37,2.0,59173,Female,Web,Member,Auto & Accessories,Car Speakers,211.0,1.0,0.3,112.0,11.2,Medium,credit_card,2018-07-24 20:41:37
2,2018-11-08,08:38:49,8.0,41066,Female,Web,Member,Auto & Accessories,Car Body Covers,117.0,5.0,0.1,31.2,3.1,Critical,credit_card,2018-11-08 08:38:49
3,2018-04-18,19:28:06,7.0,50741,Female,Web,Member,Auto & Accessories,Car & Bike Care,118.0,1.0,0.3,26.2,2.6,High,credit_card,2018-04-18 19:28:06
4,2018-08-13,21:18:39,9.0,53639,Female,Web,Member,Auto & Accessories,Tyre,250.0,1.0,0.3,160.0,16.0,Critical,credit_card,2018-08-13 21:18:39


In [6]:
# Display dataset overview
print(f"📊 Dataset Overview:")
print(f"{ecommerce_data.shape[0]:,} transactions from {ecommerce_data['Customer_Id'].nunique():,} customers")
print(f"📅 Date range: {ecommerce_data['OderedDateTime'].min().strftime('%Y-%m-%d')} to {ecommerce_data['OderedDateTime'].max().strftime('%Y-%m-%d')}")
print(f"💰 Total revenue: ${ecommerce_data['Sales'].sum():,.2f}")

📊 Dataset Overview:
51,290 transactions from 38,997 customers
📅 Date range: 2018-01-01 to 2018-12-30
💰 Total revenue: $7,813,411.00


## 1️⃣ Basic User Retention Analysis

Let's start with the most fundamental cohort analysis: tracking how many users return over time.

**Key Parameters:**
- `date_column`: Column with transaction dates
- `user_column`: Column with customer identifiers  
- `cohort_period`: How to group cohorts ('M' = monthly cohorts)
- `period_duration`: Length of analysis periods (30 days)

### Pivot Format (Default)

In [10]:
# Generate basic user retention cohorts
basic_cohorts = generate_cohort_data(
    data=ecommerce_data,
    date_column='OderedDateTime',
    user_column='Customer_Id',
    cohort_period='M',        # Monthly cohorts
    period_duration=90        # 30-day analysis periods
)

print("🎯 Basic User Retention Cohorts (Pivot Format):")
basic_cohorts

🎯 Basic User Retention Cohorts (Pivot Format):


period_number,0,1,2,3,4
cohort_period,,,,,
2018-01-01,2473,367,383,374,1
2018-02-01,2116,303,340,183,0
2018-03-01,2701,386,447,93,0
2018-04-01,3485,493,481,0,0
2018-05-01,4624,734,421,0,0
2018-06-01,3367,517,120,0,0
2018-07-01,4156,559,0,0,0
2018-08-01,3171,273,0,0,0
2018-09-01,3360,103,0,0,0


In [11]:
# Same analysis in long format
long_format_cohorts = generate_cohort_data(
    data=ecommerce_data,
    date_column='OderedDateTime',
    user_column='Customer_Id',
    cohort_period='M',
    period_duration=90,
    output_format='long'      # Long format for different use cases
)

print("📋 Same Data in Long Format (first 10 rows):")
long_format_cohorts.head(10)

📋 Same Data in Long Format (first 10 rows):


,cohort_period,period_number,metric_value
0,2018-01-01,0,2473
1,2018-01-01,1,367
2,2018-01-01,2,383
3,2018-01-01,3,374
4,2018-01-01,4,1
5,2018-02-01,0,2116
6,2018-02-01,1,303
7,2018-02-01,2,340
8,2018-02-01,3,183
9,2018-02-01,4,0


## Interpreting Cohort Results

### 🔍 How to Read Cohort Tables

**Table Structure:**
- **Rows (Index)**: Each cohort represents users acquired in the same period
- **Columns**: Time periods after acquisition (0, 1, 2, 3, etc.)

**Key Insights:**
- **Period 0**: Initial cohort size (all users acquired in that period)
- **Period 1**: Users who returned after 1 period (30 days)
- **Period 2**: Users who returned after 2 periods (60 days)
- **Declining numbers**: Normal pattern showing user churn over time

**Analysis Tips:**
- Compare **rows** to evaluate different acquisition periods
- Compare **columns** to understand retention patterns over time

## 2️⃣ Retention Rate Analysis

Absolute numbers can be hard to compare across cohorts of different sizes. Retention rates show percentages, making it easier to identify trends.

### Converting to Percentages

In [12]:
# Calculate retention rates as percentages
retention_rates = generate_cohort_data(
    data=ecommerce_data,
    date_column='OderedDateTime',
    user_column='Customer_Id',
    cohort_period='M',
    period_duration=90,
    calculate_retention_rate=True  # This converts to percentages
)

print("📈 User Retention Rates (%):") 
retention_rates

📈 User Retention Rates (%):


period_number,0,1,2,3,4
cohort_period,,,,,
2018-01-01,100.0,14.84,15.490000,15.12,0.04
2018-02-01,100.0,14.32,16.070000,8.65,0.00
2018-03-01,100.0,14.29,16.549999,3.44,0.00
2018-04-01,100.0,14.15,13.800000,0.00,0.00
2018-05-01,100.0,15.87,9.100000,0.00,0.00
2018-06-01,100.0,15.35,3.560000,0.00,0.00
2018-07-01,100.0,13.45,0.000000,0.00,0.00
2018-08-01,100.0,8.61,0.000000,0.00,0.00
2018-09-01,100.0,3.07,0.000000,0.00,0.00


### 🎨 Formatting for Better Presentation

Let's create a utility function to format our results nicely:

In [10]:
from typing import Any, Optional

def format_cohort_values(x, digits=1):
    """
    Format cohort values for better presentation.
    
    :param x: Value to format
    :param digits: Decimal places for floats (0 = integers)
    :return: Formatted value
    """
    if isinstance(x, float):
        if digits == 0:
            return int(x)
        if x.is_integer():
            return int(x)
        return round(x, digits)
    return x

# Apply formatting to retention rates
formatted_retention = retention_rates.apply(
    lambda col: col.map(lambda x: format_cohort_values(x, digits=1))
)

print("✨ Formatted Retention Rates:")
formatted_retention

✨ Formatted Retention Rates:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,100,37.1,35.2,39.0,34.4,37.8,37.5,34.9,34.8,38.6,40.3,50.2,19.2
2011-01-01,100,25.4,28.0,30.2,29.9,29.5,24.2,26.8,33.0,34.2,31.1,3.1,0.0
2011-02-01,100,22.1,24.2,29.0,21.0,26.3,24.0,29.5,28.2,26.3,3.2,0.0,0.0
2011-03-01,100,18.4,24.1,22.7,20.2,20.2,25.0,26.6,22.0,4.6,0.0,0.0,0.0
2011-04-01,100,26.8,18.1,22.1,18.1,23.1,24.8,24.1,3.3,0.0,0.0,0.0,0.0
2011-05-01,100,20.1,15.1,20.1,24.7,21.9,28.0,2.2,0.0,0.0,0.0,0.0,0.0
2011-06-01,100,16.6,20.9,25.1,28.1,29.4,3.4,0.0,0.0,0.0,0.0,0.0,0.0
2011-07-01,100,17.8,20.9,24.1,24.1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-08-01,100,21.0,29.9,23.4,1.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3️⃣ Flexible Time Periods

### Period Duration Shortcuts

RepeatRadar supports convenient shortcuts for common time periods:

- **`'D'`** = Daily (1 day)
- **`'W'`** = Weekly (7 days)  
- **`'M'`** = Monthly (~30 days)
- **`'Q'`** = Quarterly (~90 days)
- **`'Y'`** = Yearly (~365 days)

### Example: Monthly Cohorts with Weekly Analysis

In [11]:
# Monthly cohorts analyzed in weekly intervals
weekly_analysis = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    cohort_period='M',        # Group users by month of acquisition
    period_duration='W'       # Analyze retention weekly
)

print("📅 Monthly Cohorts with Weekly Analysis:")
weekly_analysis

📅 Monthly Cohorts with Weekly Analysis:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53
cohort_period,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2010-12-01,948,165,62,51,102,151,118,121,112,125,116,103,108,114,123,139,123,130,121,100,104,118,148,125,118,113,126,125,117,104,107,121,122,117,96,117,122,110,123,126,110,123,129,137,117,117,153,165,182,170,203,159,121,18
2011-01-01,421,32,32,28,27,36,29,40,29,33,40,29,31,31,33,35,52,31,51,39,34,32,34,40,36,29,31,34,39,27,33,37,33,29,42,40,41,37,50,35,43,47,43,59,50,42,34,15,3,0,0,0,0,0
2011-02-01,380,51,22,18,22,26,25,23,21,30,24,27,25,23,37,27,31,23,19,22,19,25,31,25,29,26,25,26,20,32,42,33,26,29,28,33,30,24,35,41,41,20,16,7,2,0,0,0,0,0,0,0,0,0
2011-03-01,440,22,28,25,24,20,30,22,37,28,32,29,30,32,31,27,27,28,32,24,24,22,24,32,25,28,29,32,33,24,27,41,40,37,34,39,39,26,14,17,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-04-01,299,26,14,15,23,24,26,21,19,17,14,11,12,20,18,23,13,14,19,18,13,13,25,21,20,15,18,25,18,25,21,27,25,14,10,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-05-01,279,25,21,16,22,16,11,10,13,12,14,14,14,17,16,14,16,19,13,13,25,24,17,13,20,15,29,26,23,17,6,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-06-01,235,15,18,13,14,13,7,11,13,15,12,13,17,23,15,13,17,17,18,24,16,29,26,30,12,10,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-07-01,191,13,13,17,8,10,13,10,8,14,8,15,9,12,16,11,16,16,15,15,10,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-08-01,167,10,17,8,14,13,7,13,16,10,19,14,12,15,15,10,5,3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### 🧠 Understanding Retention Rates

Retention rates show the percentage of users from each cohort who return in subsequent periods:

- **Period 0** = Always 100% (all users by definition)
- **Period 1** = % of original users returning in period 1
- **Period 2** = % of original users returning in period 2
- And so on...

**Business Insights:**
- Higher retention rates = more loyal customers
- Stable rates across periods = predictable business
- Seasonal patterns might show in different cohorts

In [12]:
# Monthly retention rates for clearer trends
monthly_retention = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    calculate_retention_rate=True,
    period_duration='M'       # Monthly analysis periods
)

print("📊 Monthly Retention Rates:")
monthly_retention.apply(
    lambda col: col.map(lambda x: format_cohort_values(x, digits=1))
)

📊 Monthly Retention Rates:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,100,37.1,35.2,39.0,34.4,37.8,37.5,34.9,34.8,38.6,40.3,50.2,19.2
2011-01-01,100,25.4,28.0,30.2,29.9,29.5,24.2,26.8,33.0,34.2,31.1,3.1,0.0
2011-02-01,100,22.1,24.2,29.0,21.0,26.3,24.0,29.5,28.2,26.3,3.2,0.0,0.0
2011-03-01,100,18.4,24.1,22.7,20.2,20.2,25.0,26.6,22.0,4.6,0.0,0.0,0.0
2011-04-01,100,26.8,18.1,22.1,18.1,23.1,24.8,24.1,3.3,0.0,0.0,0.0,0.0
2011-05-01,100,20.1,15.1,20.1,24.7,21.9,28.0,2.2,0.0,0.0,0.0,0.0,0.0
2011-06-01,100,16.6,20.9,25.1,28.1,29.4,3.4,0.0,0.0,0.0,0.0,0.0,0.0
2011-07-01,100,17.8,20.9,24.1,24.1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-08-01,100,21.0,29.9,23.4,1.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4️⃣ Value-Based Cohort Analysis

Beyond counting users, analyze the **financial value** each cohort generates over time.

### Revenue Analysis

In [13]:
# Analyze total revenue per cohort over time
revenue_cohorts = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='TotalPrice',    # Analyze revenue instead of user counts
    aggregation_function='sum',   # Sum up the revenue
    cohort_period='M',
    period_duration='W'           # Weekly revenue tracking
)

print("💰 Revenue Cohorts (Weekly Analysis):")
revenue_cohorts.apply(
    lambda col: col.map(lambda x: format_cohort_values(x, digits=0))
)

💰 Revenue Cohorts (Weekly Analysis):


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53
cohort_period,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2010-12-01,445345,72703,34858,26361,77739,60387,71351,47597,59587,70115,52538,41630,44089,64878,83836,78917,64705,50382,44345,46291,54938,54198,108821,59956,70739,56310,79767,72333,55081,77962,48669,70192,85201,115132,54465,67872,59621,72274,78366,97813,93383,149378,100115,133845,86074,108776,91404,117709,106180,128878,114107,88689,92181,13172
2011-01-01,197005,5099,5832,7990,11185,29096,7885,27086,9575,12604,13512,7878,12439,8614,15084,14029,18853,8533,35054,12333,14037,11142,35657,14435,11415,22854,13813,13968,17116,8638,10006,37533,13944,14798,18850,13619,19356,18431,43986,15054,25326,17786,14108,43165,30366,16026,12082,9488,1780,0,0,0,0,0
2011-02-01,137460,9355,4173,3277,7963,9616,6273,6059,9942,9793,11209,10188,7678,9099,12955,7637,12899,6205,8238,5098,5875,8106,8510,10147,8760,9644,11768,8888,6418,10598,21027,14910,11368,9110,11397,14912,10958,10461,15287,15963,13518,7433,4744,3000,871,0,0,0,0,0,0,0,0,0
2011-03-01,183265,3933,4812,5652,7709,5599,10938,5128,14585,12673,12360,9914,10476,9021,12496,9036,8303,10038,11395,9262,8604,8106,9929,12722,9944,12312,14633,13010,13428,12479,21799,14125,16947,10529,14657,15295,13075,7038,3504,3181,323,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-04-01,113312,4914,4866,3178,7331,7641,8383,5213,5220,7165,4679,4683,4257,5693,5173,8488,3418,7005,6031,5606,3293,8094,7921,7795,5050,3908,8777,9259,6027,8982,5096,9348,6598,4504,2750,1013,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-05-01,113041,1801,2640,2794,5615,3699,3214,3484,7007,3803,3806,6714,3065,3954,3741,3421,6847,4468,4670,5350,12424,10070,5344,3971,6028,4311,9321,7405,9061,8047,1445,424,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-06-01,90879,1562,2105,3918,1726,5039,1174,3994,2790,3211,2151,4126,3818,7692,10871,4732,7288,5327,8299,7590,3721,8849,12807,11321,2575,3379,1594,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-07-01,62975,1697,1829,5570,1436,4074,4133,2154,1600,5272,3639,4087,2204,3732,5779,2638,4034,6691,3827,3228,2120,1553,-11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011-08-01,75678,235,2047,9780,5880,6032,4227,8639,9400,4016,16776,5472,17188,10287,4050,2202,597,1029,83,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 5️⃣ Advanced Aggregation Functions

RepeatRadar supports various ways to analyze your data:

### Available Functions:
- **`sum`** 📊 Total values (revenue, quantities, points)
- **`mean`** 📈 Average values (AOV, session duration, ratings)
- **`median`** 🎯 Median values (less affected by outliers)
- **`count`** 🔢 Count of records (transactions, events)
- **`nunique`** 🔍 Count of unique values (products, categories)
- **`min`/`max`** ⬆️⬇️ Minimum/maximum values

Let's explore practical examples!

### Average Order Value (AOV) Analysis

To calculate proper AOV, let's first group transactions into complete orders:

In [14]:
# Group transactions by order to calculate proper AOV
order_data = ecommerce_data.groupby(['CustomerID', 'InvoiceDateTime', 'InvoiceNo']).agg(
    TotalPrice=('TotalPrice', 'sum')
).reset_index()

print(f"📦 Order-level data: {order_data.shape[0]:,} orders")
print(f"💵 Average order value: ${order_data['TotalPrice'].mean():.2f}")
order_data.head()

📦 Order-level data: 22,221 orders
💵 Average order value: $372.55


,CustomerID,InvoiceDateTime,InvoiceNo,TotalPrice
0,12346,2011-01-18 10:01:00,541431,77183.60
1,12346,2011-01-18 10:17:00,C541433,-77183.60
2,12347,2010-12-07 14:57:00,537626,711.79
3,12347,2011-01-26 14:30:00,542237,475.39
4,12347,2011-04-07 10:43:00,549222,636.25


In [15]:
# Calculate average order value per cohort over time
aov_cohorts = generate_cohort_data(
    data=order_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='TotalPrice',
    aggregation_function='mean',   # Average order value
    period_duration='M'
)

print("💵 Average Order Value by Cohort:")
aov_cohorts.round(2)

💵 Average Order Value by Cohort:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,326.43,399.11,367.94,422.48,334.17,381.41,442.00,492.34,442.18,595.43,540.95,445.48,471.58
2011-01-01,352.12,443.15,285.75,298.58,374.81,474.32,332.30,492.71,373.91,406.69,426.51,598.84,0.00
2011-02-01,289.40,265.25,296.60,280.50,258.98,265.26,343.99,356.94,363.97,320.73,291.96,0.00,0.00
2011-03-01,327.41,248.20,309.95,274.84,326.85,340.82,400.17,344.34,293.19,154.23,0.00,0.00,0.00
2011-04-01,301.46,254.30,334.85,290.42,296.93,331.22,312.37,265.40,189.86,0.00,0.00,0.00,0.00
2011-05-01,295.30,281.35,308.72,232.05,347.36,291.29,294.49,207.67,0.00,0.00,0.00,0.00,0.00
2011-06-01,292.68,256.75,192.83,402.90,291.28,308.78,290.06,0.00,0.00,0.00,0.00,0.00,0.00
2011-07-01,271.74,235.56,303.09,253.23,235.17,97.07,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2011-08-01,379.88,438.13,551.88,470.01,278.23,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [16]:
# Median order value (less affected by outliers)
median_aov = generate_cohort_data(
    data=order_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID', 
    value_column='TotalPrice',
    aggregation_function='median', # Median order value
    period_duration='M'
)

print("📊 Median Order Value by Cohort:")
median_aov.round(2)

📊 Median Order Value by Cohort:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,207.00,227.44,228.60,245.64,231.76,245.90,226.33,236.26,275.37,295.04,299.26,284.04,306.7
2011-01-01,212.15,210.80,201.22,226.70,250.60,250.23,232.98,273.72,293.00,241.53,270.74,397.02,0.0
2011-02-01,207.71,249.60,235.14,221.18,208.50,217.60,291.13,266.25,299.85,215.84,282.11,0.00,0.0
2011-03-01,233.42,222.77,236.46,213.80,275.24,271.33,316.83,300.76,245.05,194.98,0.00,0.00,0.0
2011-04-01,246.04,205.70,269.91,242.84,249.18,259.20,226.00,198.40,154.87,0.00,0.00,0.00,0.0
2011-05-01,211.73,213.14,185.70,180.12,243.02,235.00,233.07,221.44,0.00,0.00,0.00,0.00,0.0
2011-06-01,189.47,188.49,158.20,250.95,223.68,244.66,315.13,0.00,0.00,0.00,0.00,0.00,0.0
2011-07-01,220.95,144.00,281.94,191.76,169.13,97.07,0.00,0.00,0.00,0.00,0.00,0.00,0.0
2011-08-01,227.00,203.52,284.18,149.05,268.23,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0


In [20]:
# Transaction Frequency Analysis
# Count how many orders each cohort makes over time
transaction_count = generate_cohort_data(
    data=order_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='InvoiceNo',
    aggregation_function='count',  # Count orders per customer per period
    period_duration='M'
)

print("🛒 Transaction Count by Cohort")
transaction_count

🛒 Transaction Count by Cohort


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,1806,675,636,738,626,782,714,676,651,771,832,1103,323
2011-01-01,619,172,178,202,196,176,182,175,210,233,250,16,0
2011-02-01,548,131,132,156,109,141,122,163,160,135,18,0,0
2011-03-01,618,123,165,155,134,126,146,193,169,29,0,0,0
2011-04-01,421,125,65,83,74,92,110,99,16,0,0,0,0
2011-05-01,413,72,59,81,90,80,120,9,0,0,0,0,0
2011-06-01,339,51,72,79,97,109,9,0,0,0,0,0,0
2011-07-01,268,49,53,66,72,2,0,0,0,0,0,0,0
2011-08-01,233,58,80,52,4,0,0,0,0,0,0,0,0


In [22]:
# This shoulb be the same as:
# Transaction Frequency Analysis
# Count how many orders each cohort makes over time
transaction_count = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='InvoiceNo',
    aggregation_function='nunique',  # Count orders per customer per period
    period_duration='M'
)

print("🛒 Transaction Count by Cohort using the original data")
transaction_count

🛒 Transaction Count by Cohort using the original data


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,1805,672,633,737,624,780,714,676,650,770,832,1103,323
2011-01-01,616,171,178,202,195,176,182,175,210,232,250,16,0
2011-02-01,545,130,132,156,109,141,122,163,160,135,18,0,0
2011-03-01,616,123,165,154,134,126,146,193,168,29,0,0,0
2011-04-01,421,124,65,83,74,92,110,99,16,0,0,0,0
2011-05-01,413,72,59,81,90,80,119,9,0,0,0,0,0
2011-06-01,339,51,72,79,97,109,9,0,0,0,0,0,0
2011-07-01,268,49,53,66,72,2,0,0,0,0,0,0,0
2011-08-01,232,58,80,52,4,0,0,0,0,0,0,0,0


## 6️⃣ Segment Analysis with Data Filtering

Analyze specific customer segments by filtering your data before cohort analysis.

### High-Value Customer Analysis

In [ ]:
# Filter for high-value customers (orders > $50)
high_value_customers = ecommerce_data[ecommerce_data['TotalPrice'] > 50]

print(f"🎯 High-value segment: {high_value_customers.shape[0]:,} transactions")
print(f"💼 Regular dataset: {ecommerce_data.shape[0]:,} transactions")
print(f"📊 High-value percentage: {(high_value_customers.shape[0]/ecommerce_data.shape[0]*100):.1f}%")

# Analyze retention for high-value customers
high_value_retention = generate_cohort_data(
    data=high_value_customers,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    calculate_retention_rate=True,
    period_duration='M'
)

print("💎 High-Value Customer Retention Rates:")
high_value_retention.apply(
    lambda col: col.map(lambda x: format_cohort_values(x, digits=1))
)

🎯 High-value segment: 26,905 transactions
💼 Regular dataset: 401,604 transactions
📊 High-value percentage: 6.7%

💎 High-Value Customer Retention Rates:


period_number,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort_period,,,,,,,,,,,,,
2010-12-01,100,33.4,31.5,34.6,28.8,32.0,31.7,30.0,29.1,33.9,35.3,51.3,20.6
2011-01-01,100,20.9,22.9,23.9,26.4,25.9,24.4,20.9,20.4,26.9,23.4,1.0,0.0
2011-02-01,100,18.4,21.2,22.9,14.5,19.0,26.3,21.8,20.7,22.4,2.2,0.0,0.0
2011-03-01,100,14.4,19.9,13.4,13.0,17.1,19.9,21.3,17.6,3.2,0.0,0.0,0.0
2011-04-01,100,16.7,16.7,16.0,14.1,15.4,14.1,15.4,1.3,0.0,0.0,0.0,0.0
2011-05-01,100,15.0,14.0,14.0,17.6,12.4,17.6,1.5,0.0,0.0,0.0,0.0,0.0
2011-06-01,100,14.5,11.4,19.1,23.7,18.3,3.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-07-01,100,18.4,14.7,16.5,16.5,0.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-08-01,100,8.0,14.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


If we compare this to the original cohort, we can see how high-value customers behave a bit differently: We cannot confirm that they coming back more often as their retention rate is lower.

### 💡 Why Filter Data?

**Strategic Benefits:**
- 🎯 **Segment-specific insights**: Different customer groups show different patterns
- 💎 **Quality focus**: High-value customers often behave differently  
- 🚀 **Targeted strategies**: Understanding segments enables personalized retention campaigns
- 📊 **Comparative analysis**: Compare filtered vs. unfiltered results for deeper insights

**Example Comparisons:**
- Higher retention in filtered data → High-value customers are more loyal
- Lower retention in filtered data → High-value customers have higher expectations
- Similar patterns → Universal behavior across segments

### Recent Cohorts Analysis

Analyze only recent cohorts to understand current trends.

In [ ]:
max_date = ecommerce_data['InvoiceDateTime'].max()
recent_cutoff = max_date - pd.DateOffset(months=6) # filter for last 6 months
recent_data = ecommerce_data[ecommerce_data['InvoiceDateTime'] >= recent_cutoff]

recent_cohorts = generate_cohort_data(
    data=recent_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    calculate_retention_rate=True,
    period_duration='W',
    cohort_period='M'
)

recent_cohorts.apply(
    lambda col: col.map(lambda x: format_cohort_values(x, digits=1))
)

period_number,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26
cohort_period,,,,,,,,,,,,,,,,,,,,,,,,,,,
2011-06-01,100,13.5,13.7,13.1,16.5,18.4,14.1,17.0,15.0,14.9,17.1,16.5,16.1,17.6,18.9,17.5,18.1,14.8,18.0,20.7,22.3,23.2,21.4,22.0,14.4,8.6,0.4
2011-07-01,100,7.6,8.1,9.1,8.2,9.1,10.6,9.5,8.8,11.6,8.5,12.8,10.1,11.6,9.8,12.1,12.4,14.8,14.3,12.5,10.0,6.9,2.8,0.0,0.0,0.0,0.0
2011-08-01,100,7.3,9.3,5.9,9.1,8.8,6.4,8.6,10.3,7.3,10.5,9.3,12.0,10.0,13.7,8.8,4.2,3.2,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-09-01,100,9.3,6.1,8.2,8.6,8.1,11.4,9.8,11.6,14.7,8.4,6.7,3.0,2.5,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-10-01,100,6.6,7.7,8.7,10.0,10.0,8.7,4.4,4.4,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-11-01,100,9.0,6.8,5.0,2.6,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-12-01,100,1.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 🎯 Conclusion

Congratulations! You've mastered RepeatRadar's cohort calculation capabilities.

### What You've Learned

**Core Concepts:**
- Generated user retention cohorts with different time periods
- Calculated retention rates and percentages for better comparison
- Used flexible aggregation functions for value-based analysis
- Applied data filtering for segment-specific insights

**Technical Skills:**
- Used `generate_cohort_data()` with various parameters and configurations
- Created both pivot and long format outputs for different use cases
- Applied custom formatting functions for better data presentation
- Analyzed revenue, transaction counts, and customer behavior patterns

**Business Applications:**
- Identified high-value customer segments and their behavior patterns
- Compared cohort performance across different time periods
- Analyzed recent trends vs. historical performance
- Created actionable insights for customer retention strategies

### Next Steps

**Advanced Analysis:**
- Combine cohort calculations with visualization techniques
- Create automated cohort reporting pipelines
- Experiment with different cohort definitions and time periods
- Develop custom metrics for your specific business needs

**Visualization & Reporting:**
- Use the `02_cohort_visualization_demo.ipynb` notebook to create compelling charts
- Export cohort data for integration with BI tools and dashboards
- Create executive summaries with key retention metrics
- Set up automated monitoring of cohort performance

**Business Strategy:**
- Use cohort insights to optimize marketing campaigns
- Identify the best acquisition channels and time periods
- Develop targeted retention programs for different customer segments
- Benchmark performance against industry standards

### Key Takeaways

**Remember the Fundamentals:**
- **Period 0** always shows acquisition size (100% retention)
- **Subsequent periods** reveal retention patterns and churn
- **Compare across cohorts** to identify trends and outliers
- **Use percentages** for fair comparison between different cohort sizes

**Best Practices:**
- Start with basic user retention before diving into value analysis
- Use appropriate time periods for your business cycle
- Filter data thoughtfully to gain segment-specific insights
- Combine absolute numbers with retention rates for complete picture

### Additional Resources

- **Visualization Tutorial**: Check out `02_cohort_visualization_demo.ipynb` for creating charts
- **Documentation**: Explore the full API documentation for advanced features
- **Real-world Applications**: Apply these techniques to your own datasets
- **Community**: Share your analyses and learn from other RepeatRadar users

Happy analyzing! 📊